# PRACTICE ONLY: Scraping all ~1,000 books from Books to Scrape

This notebook practises **pagination** and **detail pages** on <https://books.toscrape.com/>, a sandbox site built for scraping practice.

Steps: import libraries, check `robots.txt`, download a page, read the HTML with BeautifulSoup, find the book cards, follow the pagination links, visit every book's detail page, and save everything into a pandas table / CSV.

First check the website rules. If a page is disallowed, do not scrape it.

## 1. Import the tools
These libraries help us download pages, read HTML and organise information in a table.

In [ ]:
import re
import time                          # Time lets us wait between website requests.
from datetime import datetime
from urllib.parse import urljoin     # urljoin builds full URLs from relative links.

import requests                      # Requests downloads a web page.
from bs4 import BeautifulSoup        # BeautifulSoup reads the HTML code.
import pandas as pd                  # Pandas puts the information into a table.

## 2. Choose the website
This is the practice site assigned to our group (Group 7). It has **50 listing pages with 20 books each = 1,000 books**.

In [ ]:
BASE_URL = "https://books.toscrape.com/"                 # The website we are studying.
START_URL = urljoin(BASE_URL, "catalogue/page-1.html")   # First listing page.
HEADERS = {"User-Agent": "B32919-Group7-UCU"}            # Identifies our student project.
DELAY = 0.5                                              # Seconds to wait between requests.

# A Session re-uses the connection, which makes ~1,050 requests much faster.
session = requests.Session()
session.headers.update(HEADERS)

def get_soup(url):
    """Download a page and return it as a BeautifulSoup object."""
    response = session.get(url, timeout=30)
    response.raise_for_status()
    # The site does not declare its encoding, so force UTF-8 (otherwise £ becomes Â£).
    response.encoding = "utf-8"
    return BeautifulSoup(response.text, "html.parser")

## 3. Check robots.txt
Read these rules and record the relevant result in the reconnaissance note. A `404` means the site publishes no robots.txt rules. Do not continue if the page we need is disallowed.

In [ ]:
robots = session.get(urljoin(BASE_URL, "robots.txt"), timeout=30)
print("robots.txt status:", robots.status_code)
if robots.status_code == 200:
    print(robots.text[:1500])
else:
    print("No robots.txt found, so no crawling restrictions are published.")

## 4. Download the first page
A successful request returns status code 200. If it fails or access is denied, stop and investigate.

In [ ]:
# This downloads the first listing page.
response = session.get(START_URL, timeout=30)
# This displays the response status code.
print(response.status_code)
# This stops the notebook if the request failed.
response.raise_for_status()
response.encoding = "utf-8"

## 5. Read the HTML
BeautifulSoup turns the raw text into a structure we can search.

In [ ]:
# This makes the HTML easier to search.
soup = BeautifulSoup(response.text, "html.parser")
# This displays the page title and a small part of the downloaded HTML.
print(soup.title.get_text(strip=True))
print(response.text[:500])

## 6. Find the book cards
Each book on a listing page sits inside an `article` element with the class `product_pod`. If zero are found, the page layout may have changed. There should be **20** on this page.

In [ ]:
# This finds the book cards in the downloaded page.
rows = soup.select("article.product_pod")
# This tells us how many cards were found.
print("Rows found:", len(rows))

## 7. Collect fields from one listing page
Each comment explains the line below it. Missing fields are left blank rather than guessed. We write this as a function so we can reuse it on every page.

In [ ]:
RATING_WORDS = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

def parse_listing_page(soup, page_url):
    """Return one dictionary per book found on a listing page."""
    books = []
    for row in soup.select("article.product_pod"):
        # The link inside the <h3> holds the full title (the visible text is shortened).
        link_tag = row.select_one("h3 a")
        title = link_tag.get("title", "") if link_tag else ""
        # Links are relative, so join them with the current page URL.
        detail_url = urljoin(page_url, link_tag["href"]) if link_tag else ""
        # The price element.
        price_tag = row.select_one("p.price_color")
        price = price_tag.get_text(strip=True) if price_tag else ""
        # The star rating is stored as a class name, e.g. "star-rating Three".
        rating_tag = row.select_one("p.star-rating")
        rating_word = ""
        if rating_tag:
            rating_word = [c for c in rating_tag.get("class", []) if c != "star-rating"][0]
        # Availability text on the listing page.
        stock_tag = row.select_one("p.availability")
        availability = stock_tag.get_text(strip=True) if stock_tag else ""
        books.append({
            "title": title,
            "price": price,
            "rating": RATING_WORDS.get(rating_word),
            "availability": availability,
            "detail_url": detail_url,
        })
    return books

# Test it on the first page only.
first_page_books = parse_listing_page(soup, START_URL)
pd.DataFrame(first_page_books).head()

## 8. Pagination: collect every listing page
Instead of guessing how many pages exist, we follow the **Next** button (`li.next a`) until it disappears. The site has 50 pages, so we expect 1,000 books.

In [ ]:
all_books = []
page_url = START_URL
page_number = 1

while page_url:
    page_soup = get_soup(page_url)
    page_books = parse_listing_page(page_soup, page_url)
    all_books.extend(page_books)
    print(f"Page {page_number}: {len(page_books)} books (total {len(all_books)})")

    # Find the Next link; if there is none, we are on the last page.
    next_tag = page_soup.select_one("li.next a")
    page_url = urljoin(page_url, next_tag["href"]) if next_tag else None
    page_number += 1
    # Be polite: wait before the next request.
    time.sleep(DELAY)

print("Total books found:", len(all_books))

## 9. Detail pages
The listing page does not show the category, UPC, tax, number of reviews or description. For those we visit each book's own page. This is about 1,000 requests, so it takes several minutes.

In [ ]:
def parse_detail_page(soup):
    """Return the extra fields found on one book's detail page."""
    # The product information table has one <th> label and one <td> value per row.
    info = {}
    for tr in soup.select("table.table-striped tr"):
        th, td = tr.find("th"), tr.find("td")
        if th and td:
            info[th.get_text(strip=True)] = td.get_text(strip=True)

    # The category is the third item in the breadcrumb (Home > Books > Category > Title).
    crumbs = soup.select("ul.breadcrumb li a")
    category = crumbs[2].get_text(strip=True) if len(crumbs) >= 3 else ""

    # The description is the <p> that follows the #product_description heading.
    desc_header = soup.select_one("#product_description")
    desc_tag = desc_header.find_next_sibling("p") if desc_header else None
    description = desc_tag.get_text(strip=True) if desc_tag else ""

    # Turn "In stock (22 available)" into the number 22.
    match = re.search(r"\d+", info.get("Availability", ""))
    stock_count = int(match.group()) if match else None

    return {
        "upc": info.get("UPC", ""),
        "category": category,
        "price_excl_tax": info.get("Price (excl. tax)", ""),
        "price_incl_tax": info.get("Price (incl. tax)", ""),
        "tax": info.get("Tax", ""),
        "stock_count": stock_count,
        "number_of_reviews": int(info["Number of reviews"]) if info.get("Number of reviews", "").isdigit() else None,
        "description": description,
    }

# Test on a single book before running all of them.
test_details = parse_detail_page(get_soup(all_books[0]["detail_url"]))
test_details

In [ ]:
scraped_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
full_books = []
failed = []

for i, book in enumerate(all_books, start=1):
    try:
        details = parse_detail_page(get_soup(book["detail_url"]))
        full_books.append({**book, **details, "scraped_at": scraped_at})
    except requests.RequestException as error:
        # Keep going if one page fails, and remember it so we can retry.
        failed.append(book["detail_url"])
        print("Failed:", book["detail_url"], error)
    if i % 100 == 0:
        print(f"{i}/{len(all_books)} detail pages done")
    time.sleep(DELAY)

print("Collected:", len(full_books), "| Failed:", len(failed))

## 10. Build the table and check it
Turn the list into a pandas table, then check the row count and look for duplicates or blanks.

In [ ]:
books_df = pd.DataFrame(full_books)
print("Rows:", len(books_df))
print("Duplicate UPCs:", books_df["upc"].duplicated().sum())
print("Books per category (top 10):")
print(books_df["category"].value_counts().head(10))
books_df.head()

## 11. Save the CSV
This saves the full dataset (about 1,000 books). `utf-8-sig` keeps the £ sign readable in Excel.

In [ ]:
books_df.to_csv("books_full.csv", index=False, encoding="utf-8-sig")
print("Saved books_full.csv with", len(books_df), "rows")